# aerial-vg: TransVG on DIOR-RSVG + direction-aware D4 augmentation

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. The dataset is downloaded straight from the authors' Google Drive into `/tmp` (no upload needed; ~5 GB, a few minutes).

Runs 4 conditions in 2 parallel phases (one per GPU):
| name | `--geo_aug` | role |
|---|---|---|
| baseline | hflip | TransVG default (reproduction) |
| d4_remap | d4_remap | proposed |
| d4_noremap | d4_noremap | control: same geometry, text not remapped |
| no_geo | none | ablation: no geometric augmentation |

Use **Save Version → Save & Run All (Commit)** so it keeps running after you close the browser (12 h limit).

Set `SMOKE = True` in the config cell for a ~20-minute end-to-end check of this notebook before the long run.

In [1]:
import time; NB_START = time.time()   # used to keep the whole run inside Kaggle's 12 h limit
REPO_URL = 'https://github.com/HonourGok/aerial-vg.git'   # used only if the code is not attached as a Kaggle dataset
import os, subprocess, glob, shutil
DATA_ROOT = '/tmp/DIOR_RSVG'   # /tmp is not saved as notebook output, so the 5 GB stay out of the 20 GB output limit

def find_dir(root, ext):
    best = None
    for d, _, files in os.walk(root):
        n = sum(f.lower().endswith(ext) for f in files)
        if n and (best is None or n > best[1]):
            best = (d, n)
    return best

if not os.path.isdir(DATA_ROOT + '/Annotations'):
    # 1) a manually uploaded Kaggle dataset, if one was added as input
    up = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/Annotations', recursive=True)]
    if up:
        DATA_ROOT = up[0]
    else:
        # 2) download from the authors' Google Drive folder (Annotations.zip 7 MB, JPEGImages.zip 5 GB)
        subprocess.run('pip -q install gdown', shell=True, check=True)
        os.makedirs('/tmp/rsvg_dl', exist_ok=True)
        subprocess.run('gdown --folder https://drive.google.com/drive/folders/1hTqtYsC6B-m4ED2ewx5oKuYZV13EoJp_ -O /tmp/rsvg_dl',
                       shell=True, check=True)
        os.makedirs(DATA_ROOT, exist_ok=True)
        for z in glob.glob('/tmp/rsvg_dl/**/*.zip', recursive=True):
            subprocess.run(f'unzip -q -o "{z}" -d /tmp/rsvg_unz', shell=True, check=True)
            os.remove(z)
        for sub, ext in (('Annotations', '.xml'), ('JPEGImages', '.jpg')):
            d, n = find_dir('/tmp/rsvg_unz', ext)
            print(sub, '<-', d, n, 'files')
            shutil.move(d, f'{DATA_ROOT}/{sub}')

print('DATA_ROOT =', DATA_ROOT, '| xml:', len(os.listdir(DATA_ROOT + '/Annotations')),
      '| jpg:', len(os.listdir(DATA_ROOT + '/JPEGImages')))
# If gdown fails with a quota / "too many users" error: download the two zips manually and upload them as a
# Kaggle dataset (Datasets -> New Dataset), add it as input, and re-run this cell.

Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1Cz4wMifO32jLmQdxtsGyJuNjSbOxnbG_
From (redirected): https://drive.google.com/uc?id=1Cz4wMifO32jLmQdxtsGyJuNjSbOxnbG_&confirm=t&uuid=acda08eb-3100-4fc3-b2f8-385bf60c126e
To: /tmp/rsvg_dl/Annotations.zip
100%|██████████| 7.77M/7.77M [00:00<00:00, 171MB/s]


Processing file 1Cz4wMifO32jLmQdxtsGyJuNjSbOxnbG_ Annotations.zip
Processing file 1pdUl2MHXqo70dbw7BdUhkW25boveDJfo JPEGImages.zip
Processing file 14rEjQ9Mp92rkw6HoDD5MbVTMLjA0Zx1G test.txt
Processing file 12W0HZdS7ri1Xf5dyB90JyPL4dPUv7ogA train.txt
Processing file 1-9-LcZtsK663xdmtW1DtYjKztEMaroNz val.txt


Downloading...
From (original): https://drive.google.com/uc?id=1pdUl2MHXqo70dbw7BdUhkW25boveDJfo
From (redirected): https://drive.google.com/uc?id=1pdUl2MHXqo70dbw7BdUhkW25boveDJfo&confirm=t&uuid=772624f5-daa5-4cde-8095-6573393ffc6d
To: /tmp/rsvg_dl/JPEGImages.zip
100%|██████████| 5.39G/5.39G [01:32<00:00, 58.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=14rEjQ9Mp92rkw6HoDD5MbVTMLjA0Zx1G
To: /tmp/rsvg_dl/test.txt
100%|██████████| 42.9k/42.9k [00:00<00:00, 81.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=12W0HZdS7ri1Xf5dyB90JyPL4dPUv7ogA
To: /tmp/rsvg_dl/train.txt
100%|██████████| 154k/154k [00:00<00:00, 6.08MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-9-LcZtsK663xdmtW1DtYjKztEMaroNz
To: /tmp/rsvg_dl/val.txt
100%|██████████| 21.9k/21.9k [00:00<00:00, 52.6MB/s]
Download completed


Annotations <- /tmp/rsvg_unz/Annotations 17402 files
JPEGImages <- /tmp/rsvg_unz/JPEGImages 17402 files
DATA_ROOT = /tmp/DIOR_RSVG | xml: 17402 | jpg: 17402


In [2]:
%cd /kaggle/working
# code: from an attached Kaggle dataset containing aerial-vg/ (works while the GitHub repo is private), else git clone
code_src = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/aerial-vg/train.py', recursive=True)]
!rm -rf aerial-vg
if code_src:
    shutil.copytree(code_src[0], '/kaggle/working/aerial-vg')
    print('code from', code_src[0])
else:
    subprocess.run(f'git clone -q {REPO_URL} aerial-vg', shell=True, check=True)
%cd /kaggle/working/aerial-vg
!(git rev-parse HEAD 2>/dev/null || md5sum train.py datasets/geo_aug.py) | tee results_commit.txt
!pip -q install -r requirements.txt
!mkdir -p checkpoints && wget -q -nc -O checkpoints/detr-r50-e632da11.pth https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch,torchvision,transformers;print(torch.__version__,torchvision.__version__,transformers.__version__)" 

/kaggle/working
code from /kaggle/input/datasets/honourgok/aerial-vg-code/aerial-vg
/kaggle/working/aerial-vg
ff198d0091861a2eb5b1b08175309684  train.py
849c9ccb4be023a5cab470a35b19fb37  datasets/geo_aug.py
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
2.11.0+cu130 0.26.0+cu130 5.20.2


In [3]:
# one-off: resize images to 640 px (the model input size) so the data loader does not decode 800 px JPEGs every epoch
if not os.path.isdir('/tmp/DIOR_RSVG_640/JPEGImages'):
    !python tools/preresize.py --src $DATA_ROOT --dst /tmp/DIOR_RSVG_640 --size 640
DATA_ROOT = '/tmp/DIOR_RSVG_640'
os.environ['DATA_ROOT'] = DATA_ROOT
os.environ['DETR'] = '/kaggle/working/aerial-vg/checkpoints/detr-r50-e632da11.pth'
# compute budget (identical for all conditions)
# measured on T4: 17.5 train samples/s -> ~26 min per epoch on the full train split, ~5 min for the full test set
SMOKE = False   # True = 2-minute runs on tiny data, to check the whole notebook end to end in Save & Run All
if SMOKE:
    os.environ.update(EPOCHS='1', LR_DROP='1', FRACTION='0.005', VAL_MAX='40', TEST_LIMIT='200', LAT='20', MAX_HOURS='1')
else:
    os.environ.update(EPOCHS='7', LR_DROP='5', FRACTION='1.0', VAL_MAX='500', TEST_LIMIT='0', LAT='200', MAX_HOURS='5.0')
os.environ.update(BS='8', IMSIZE='640', SEED='13')
print('SMOKE =', SMOKE, {k: os.environ[k] for k in ('EPOCHS', 'LR_DROP', 'FRACTION', 'VAL_MAX', 'TEST_LIMIT', 'MAX_HOURS')})
!python tools/dataset_stats.py --data_root $DATA_ROOT --split_root splits --out results/dataset_stats.json

resized 17402 images, rewrote 17402 xml files -> /tmp/DIOR_RSVG_640
SMOKE = False {'EPOCHS': '7', 'LR_DROP': '5', 'FRACTION': '1.0', 'VAL_MAX': '500', 'TEST_LIMIT': '0', 'MAX_HOURS': '5.0'}
train {'n_expressions': 26991, 'n_images': 14730, 'pct_with_direction_word': 62.16887110518321, 'direction_word_counts': {'left': 6890, 'right': 7137, 'top': 2076, 'bottom': 2288, 'upper': 3854, 'lower': 3872, 'leftmost': 1, 'above': 548, 'below': 618}, 'mean_words': 7.439257530287874}
val {'n_expressions': 3829, 'n_images': 3465, 'pct_with_direction_word': 63.59362757900235, 'direction_word_counts': {'left': 1027, 'right': 1007, 'top': 315, 'bottom': 346, 'upper': 564, 'lower': 582, 'above': 79, 'below': 81}, 'mean_words': 7.523896578741185}
test {'n_expressions': 7500, 'n_images': 6102, 'pct_with_direction_word': 62.25333333333333, 'direction_word_counts': {'left': 1938, 'right': 1995, 'top': 575, 'bottom': 632, 'upper': 1097, 'lower': 1059, 'above': 157, 'below': 157}, 'mean_words': 7.46613333333

In [4]:
# FIX (diagnosed 2026-11-10): the post-LN fusion transformer collapsed to one constant box for every input (loss stuck ~1.85, all 4 runs).
# Pre-LN fusion + 500-iter linear LR warmup + grad-clip 0.1 fixes it (1 epoch full data: val Pr@0.5 26% vs ~2% before). Applied identically to all conditions.
import re; p='/kaggle/working/aerial-vg/'
s=open(p+'models/vl_transformer.py').read(); i=s.index('def build_vl_transformer'); t=s[i:]; assert t.count('normalize_before=False')==1; open(p+'models/vl_transformer.py','w').write(s[:i]+t.replace('normalize_before=False',"normalize_before=bool(getattr(args, 'vl_prenorm', 0))"))
s=open(p+'train.py').read(); m=re.search(r"\n( *)parser\.add_argument\('--amp'", s); ind=m.group(1); s=s[:m.start()]+"\n"+ind+"parser.add_argument('--vl_prenorm', default=1, type=int, help='pre-LN fusion transformer (post-LN collapses to a constant box)')\n"+ind+"parser.add_argument('--warmup_iters', default=0, type=int, help='linear LR warmup iterations in epoch 0')"+s[m.start():]; open(p+'train.py','w').write(s)
s=open(p+'engine.py').read(); m=re.search(r"\n( *)img_data, text_data, target = batch\n", s); ind=m.group(1); add=ind+"it = getattr(args, '_it', 0); args._it = it + 1\n"+ind+"if epoch == 0 and it < getattr(args, 'warmup_iters', 0):\n"+ind+"    for pg in optimizer.param_groups:\n"+ind+"        pg['lr'] = pg.setdefault('initial_lr', pg['lr']) * (it + 1) / args.warmup_iters\n"; open(p+'engine.py','w').write(s[:m.end()]+add+s[m.end():])
s=open(p+'scripts/run_one.sh').read(); assert s.count('python train.py')==1; open(p+'scripts/run_one.sh','w').write(s.replace('python train.py','python train.py --vl_prenorm 1 --warmup_iters 500 --clip_max_norm 0.1'))
!cd /kaggle/working/aerial-vg && python -m py_compile train.py engine.py models/vl_transformer.py && grep -n "vl_prenorm\|warmup_iters\|clip_max_norm 0.1" models/vl_transformer.py train.py engine.py scripts/run_one.sh
# Time budget: remaining GPU quota is ~10 h, so the whole commit must finish well inside it (and inside Kaggle's 12 h): target <= 9.3 h total.
RUN_BUDGET_H = 9.3
if not SMOKE: os.environ['MAX_HOURS'] = '4.2'
print('RUN_BUDGET_H =', RUN_BUDGET_H, '| phase-1 MAX_HOURS =', os.environ['MAX_HOURS'])

models/vl_transformer.py:139:        normalize_before=bool(getattr(args, 'vl_prenorm', 0)),
train.py:127:    parser.add_argument('--vl_prenorm', default=1, type=int, help='pre-LN fusion transformer (post-LN collapses to a constant box)')
train.py:128:    parser.add_argument('--warmup_iters', default=0, type=int, help='linear LR warmup iterations in epoch 0')
engine.py:31:        if epoch == 0 and it < getattr(args, 'warmup_iters', 0):
engine.py:33:                pg['lr'] = pg.setdefault('initial_lr', pg['lr']) * (it + 1) / args.warmup_iters
scripts/run_one.sh:19:CUDA_VISIBLE_DEVICES=$GPU python train.py --vl_prenorm 1 --warmup_iters 500 --clip_max_norm 0.1 \
RUN_BUDGET_H = 9.3 | phase-1 MAX_HOURS = 4.2


In [5]:
# Speed check (~3 min, interactive sessions only; skipped in Save & Run All)
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') != 'Batch':
    !CUDA_VISIBLE_DEVICES=0 python tools/benchmark.py --dataset dior_rsvg --data_root $DATA_ROOT --split_root splits \
        --detr_model $DETR --imsize 640 --max_query_len 40 --batch_size 8 --amp --num_workers 4 --iters 40 \
        --aug_crop --aug_scale --aug_translate --geo_aug d4_remap --train_fraction 0.5 --val_max 500 2>&1 | grep -v Warning

In [6]:
# Phase 1: baseline (GPU0) || proposed (GPU1)
# Jobs are started with subprocess.Popen and waited on directly; a failing job stops the cell within ~20 s.
import sys, importlib
sys.path.insert(0, 'scripts'); import launch; importlib.reload(launch)
# strict (stop on first failure) only in SMOKE; in the long run a failure of one job must not kill the other
launch.run_parallel([('baseline', 'hflip', 0), ('d4_remap', 'd4_remap', 1)], strict=SMOKE)

started baseline (geo_aug=hflip, GPU 0), pid 118
started d4_remap (geo_aug=d4_remap, GPU 1), pid 119
[0.01 h] baseline: - UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[0.01 h] d4_remap: - UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[0.17 h] baseline: Epoch: [0]  [1150/3373]  eta: 0:19:07  lr: 0.000100  loss: 1.1588 (1.6359)  loss_bbox: 0.3618 (0.5813)  loss_giou: 0.7970 (1.0546)  time: 0.4915  data: 0.0031  
[0.17 h] d4_remap: Epoch: [0]  [1150/3373]  eta: 0:19:09  lr: 0.000100  loss: 1.0623 (1.6461)  loss_bbox: 0.3044 (0.5880)  loss_giou: 0.7579 (1.0580)  time: 0.5028  data: 0.0047  
[0.34 h] baseline: Epoch: [0]  [2320/3373]  eta: 0:09:01  lr: 0.000100  loss: 1.4243 (1.5186)  loss_bbox: 0.4189 (0.5009)  loss_giou: 1.0054 (1.0177)  time: 0.5022  data: 0.0034  
[0.34 h] d4_remap: Epoch: [0]  [2320/3373]  eta: 0:09:02  lr: 0.000100  loss: 1.3557 (1.5269)  

{'baseline': 0, 'd4_remap': 0}

In [7]:
# Phase 2: control (GPU0) || ablation without geometric augmentation (GPU1)
# Time guard: everything must finish before Kaggle's 12 h limit, otherwise no output is saved.
# Runs stop after the epoch that crosses MAX_HOURS (one epoch ~0.5 h) and are then evaluated (~0.2 h).
elapsed = (time.time() - NB_START) / 3600
if not SMOKE:
    os.environ['MAX_HOURS'] = '%.2f' % max(0.5, min(5.0, RUN_BUDGET_H - elapsed - 0.5 - 0.3))
print('elapsed %.2f h, phase-2 MAX_HOURS = %s' % (elapsed, os.environ['MAX_HOURS']))
launch.run_parallel([('d4_noremap', 'd4_noremap', 0), ('no_geo', 'none', 1)], strict=SMOKE)

elapsed 3.66 h, phase-2 MAX_HOURS = 4.84
started d4_noremap (geo_aug=d4_noremap, GPU 0), pid 1163
started no_geo (geo_aug=none, GPU 1), pid 1164
[0.01 h] d4_noremap: Epoch: [0]  [   0/3373]  eta: 3:17:24  lr: 0.000000  loss: 1.9299 (1.9299)  loss_bbox: 0.9399 (0.9399)  loss_giou: 0.9900 (0.9900)  time: 3.5117  data: 1.3282  
[0.01 h] no_geo: Epoch: [0]  [   0/3373]  eta: 3:24:04  lr: 0.000000  loss: 1.9400 (1.9400)  loss_bbox: 0.9534 (0.9534)  loss_giou: 0.9866 (0.9866)  time: 3.6302  data: 0.7678  
[0.17 h] d4_noremap: Epoch: [0]  [1170/3373]  eta: 0:19:00  lr: 0.000100  loss: 1.4849 (1.7992)  loss_bbox: 0.4996 (0.6543)  loss_giou: 0.9852 (1.1449)  time: 0.5051  data: 0.0029  
[0.17 h] no_geo: Epoch: [0]  [1160/3373]  eta: 0:19:11  lr: 0.000100  loss: 1.6006 (1.6386)  loss_bbox: 0.4436 (0.5850)  loss_giou: 1.1570 (1.0537)  time: 0.5035  data: 0.0036  
[0.34 h] d4_noremap: Epoch: [0]  [2330/3373]  eta: 0:08:59  lr: 0.000100  loss: 1.4514 (1.7442)  loss_bbox: 0.4960 (0.6019)  loss_giou:

{'d4_noremap': 0, 'no_geo': 0}

In [8]:
# Collect: comparison over the runs that produced metrics, figure, training logs, zip
runs = [r for r in ['baseline', 'd4_remap', 'd4_noremap', 'no_geo'] if os.path.exists(f'results/{r}/metrics.json')]
print('runs with results:', runs)
if 'baseline' in runs and len(runs) > 1:
    subprocess.run(['python', 'compare.py', '--runs'] + [f'{r}=results/{r}' for r in runs] +
                   ['--ref', 'baseline', '--out', 'results/comparison'], check=False)
if 'baseline' in runs and 'd4_remap' in runs:
    subprocess.run(['python', 'tools/viz_examples.py', '--n', '8', '--a', 'results/baseline', '--b', 'results/d4_remap',
                    '--data_root', DATA_ROOT, '--out', 'results/figures/baseline_vs_d4remap.png'], check=False)
for r in ['baseline', 'd4_remap', 'd4_noremap', 'no_geo']:
    if os.path.exists(f'outputs/{r}/log.txt'):
        os.makedirs(f'results/{r}', exist_ok=True)
        shutil.copy(f'outputs/{r}/log.txt', f'results/{r}/train_log.txt')
os.makedirs('results/logs', exist_ok=True)
for f in glob.glob('logs/*.log'):
    shutil.copy(f, 'results/logs/')
shutil.copy('results_commit.txt', 'results/')
print('total hours: %.2f' % ((time.time() - NB_START) / 3600))
!cd /kaggle/working && rm -f results.zip && zip -qr results.zip aerial-vg/results && ls -lh results.zip
!du -sh /kaggle/working/aerial-vg/outputs

runs with results: ['baseline', 'd4_remap', 'd4_noremap', 'no_geo']
| Subset | n | baseline Pr@0.5 | d4_remap Pr@0.5 | d4_noremap Pr@0.5 | no_geo Pr@0.5 | Δ d4_remap vs baseline [95% CI], p | Δ d4_noremap vs baseline [95% CI], p | Δ no_geo vs baseline [95% CI], p |
|---|---|---|---|---|---|---|---|---|
| all | 7500 | 46.44 | 48.33 | 28.23 | 45.61 | +1.89 [+1.15, +2.61], p=2.34e-07 | -18.21 [-19.19, -17.27], p=2.4e-287 | -0.83 [-1.51, -0.09], p=0.0209 |
| with_direction | 4669 | 38.51 | 40.46 | 11.89 | 37.44 | +1.95 [+1.03, +2.87], p=6.36e-05 | -26.62 [-28.01, -25.29], p=3.93e-305 | -1.07 [-1.99, -0.13], p=0.028 |
| no_direction | 2831 | 59.52 | 61.32 | 55.17 | 59.10 | +1.80 [+0.71, +2.93], p=0.00117 | -4.34 [-5.58, -3.21], p=3.82e-13 | -0.42 [-1.41, +0.57], p=0.437 |
| horizontal_words | 3475 | 37.15 | 38.94 | 11.42 | 36.35 | +1.78 [+0.69, +2.85], p=0.00144 | -25.73 [-27.22, -24.26], p=5.59e-220 | -0.81 [-1.81, +0.26], p=0.151 |
| vertical_words | 3267 | 34.56 | 36.85 | 10.81 | 33.43 |